# Raman Spectrum Plotting and Comparison

This notebook analyzes mode-resolved Raman activities calculated for Li-deficient Ta-doped LLZO garnet structures.

The input files are `vasp_raman.dat` files stored in `data/processed/`. Each file contains discrete phonon-mode Raman activities. The notebook applies Gaussian broadening to construct continuous simulated Raman spectra and compares spectra across:

- Li content: Li5, Li5.5, and Li6.5
- molecular dynamics snapshot time: 50 ps, 75 ps, and 100 ps

The analysis is based on processed outputs from the original first-principles Raman workflow. The original finite-displacement Raman helper scripts are not included in this repository.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# The notebook is expected to be run from the notebooks/ directory.
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "processed"
FIGURE_DIR = PROJECT_ROOT / "figures" / "raman"

FIGURE_DIR.mkdir(parents=True, exist_ok=True)

systems = {
    "Li6p5": {
        "label": r"Li$_{6.5}$La$_3$Zr$_{1.5}$Ta$_{0.5}$O$_{12}$",
        "folders": {
            "50 ps": "Li6p5_50ps",
            "75 ps": "Li6p5_75ps",
            "100 ps": "Li6p5_100ps",
        },
    },
    "Li5p5": {
        "label": r"Li$_{5.5}$La$_3$Zr$_{1.5}$Ta$_{0.5}$O$_{11.5}$",
        "folders": {
            "50 ps": "Li5p5_50ps",
            "75 ps": "Li5p5_75ps",
            "100 ps": "Li5p5_100ps",
        },
    },
    "Li5": {
        "label": r"Li$_5$La$_3$Zr$_{1.5}$Ta$_{0.5}$O$_{11.25}$",
        "folders": {
            "50 ps": "Li5_50ps",
            "75 ps": "Li5_75ps",
            "100 ps": "Li5_100ps",
        },
    },
}

raman_range = (200, 600)
low_frequency_window = (200, 300)

## Input file format

Each processed folder contains a `vasp_raman.dat` file with the following format:

```text
# mode    freq(cm-1)    alpha    beta2    activity

In [ ]:
class SimulatedRaman:
    """Load and broaden mode-resolved Raman activities from a vasp_raman.dat file."""

    def __init__(self, path):
        self.path = Path(path)
        self.file_path = self.path / "vasp_raman.dat"

        if not self.file_path.exists():
            raise FileNotFoundError(f"Could not find Raman file: {self.file_path}")

        self.data = pd.read_csv(
            self.file_path,
            comment="#",
            delim_whitespace=True,
            names=["mode", "frequency_cm1", "alpha", "beta2", "activity"],
        )

        self.data = self.data.sort_values("frequency_cm1").reset_index(drop=True)

    def smeared_spectrum(
        self,
        x=None,
        broadening=20.0,
        normalize=True,
        frequency_range=(0, 900),
        n_points=2000,
    ):
        """Return a Gaussian-broadened Raman spectrum."""
        if x is None:
            x = np.linspace(frequency_range[0], frequency_range[1], n_points)

        y = np.zeros_like(x, dtype=float)

        for _, row in self.data.iterrows():
            frequency = row["frequency_cm1"]
            activity = row["activity"]
            gaussian = np.exp(-((x - frequency) ** 2) / (2 * broadening**2))
            y += activity * gaussian

        if normalize and np.max(y) > 0:
            y = y / np.max(y)

        return x, y

    def area_between(self, lower, upper, broadening=20.0, normalize=True):
        """Calculate area under the broadened spectrum between two frequencies."""
        x, y = self.smeared_spectrum(
            broadening=broadening,
            normalize=normalize,
            frequency_range=(lower, upper),
            n_points=1000,
        )
        return np.trapz(y, x)

    def modes_in_range(self, lower, upper):
        """Return discrete Raman-active modes within a frequency range."""
        mask = (self.data["frequency_cm1"] >= lower) & (self.data["frequency_cm1"] <= upper)
        return self.data.loc[mask].copy()

In [ ]:
raman_data = {}

for system_name, system_info in systems.items():
    raman_data[system_name] = {}

    for time_label, folder_name in system_info["folders"].items():
        folder_path = DATA_DIR / folder_name
        raman_data[system_name][time_label] = SimulatedRaman(folder_path)

print("Loaded Raman datasets:")
for system_name, time_data in raman_data.items():
    for time_label, obj in time_data.items():
        n_modes = len(obj.data)
        f_min = obj.data["frequency_cm1"].min()
        f_max = obj.data["frequency_cm1"].max()
        print(f"{system_name:5s} | {time_label:6s} | {n_modes:3d} modes | {f_min:8.2f}–{f_max:8.2f} cm^-1")

In [ ]:
summary_rows = []

for system_name, time_data in raman_data.items():
    for time_label, obj in time_data.items():
        modes_200_600 = obj.modes_in_range(*raman_range)
        modes_200_300 = obj.modes_in_range(*low_frequency_window)

        summary_rows.append(
            {
                "system": system_name,
                "time": time_label,
                "n_total_modes": len(obj.data),
                "min_frequency_cm1": obj.data["frequency_cm1"].min(),
                "max_frequency_cm1": obj.data["frequency_cm1"].max(),
                "n_modes_200_600": len(modes_200_600),
                "n_modes_200_300": len(modes_200_300),
            }
        )

summary_df = pd.DataFrame(summary_rows)
summary_df

## Data check

The table above summarizes the number of Raman-active modes available for each Li content and MD snapshot time.

The main analysis focuses on the 200–600 cm⁻¹ region because this range was used in the original study to compare Li–O-related vibrational features. A narrower 200–300 cm⁻¹ window is also evaluated because the original analysis observed intensity changes in this lower-frequency region across Li contents.